# Word Embeddings : le modèle Word2Vec

## Imports

In [1]:
import sys

from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec

import nltk
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

## Chargement et traitement des phrases du corpus

### Création d'un objet qui *streame* les lignes d'un fichier pour économiser de la RAM

In [2]:
class MySentences(object):
    """Tokenize and Lemmatize sentences"""
    def __init__(self, filename):
        self.filename = filename

    def __iter__(self):
        for line in open(self.filename, encoding='utf-8', errors="backslashreplace"):
            yield [unidecode(w.lower()) for w in wordpunct_tokenize(line)]

In [3]:
infile = f"../../data/sents.txt"
sentences = MySentences(infile)

### Détection des bigrams

Article intéressant sur le sujet : https://towardsdatascience.com/word2vec-for-phrases-learning-embeddings-for-more-than-one-word-727b6cf723cf

In [4]:
bigram_phrases = Phrases(sentences)

L'object `phrases` peut être vu comme un large dictionnaire d'expressions multi-mots associées à un score, le *PMI-like scoring*. Ce dictionnaire est construit par un apprentissage sur base d'exemples.
Voir les références ci-dessous :
- https://arxiv.org/abs/1310.4546
- https://en.wikipedia.org/wiki/Pointwise_mutual_information

In [5]:
type(bigram_phrases.vocab)

dict

Il contient de nombreuses clés qui sont autant de termes observés dans le corpus

In [6]:
len(bigram_phrases.vocab.keys())

4310398

Prenons une clé au hasard :

In [7]:
key_ = list(bigram_phrases.vocab.keys())[144]
print(key_)

1q


Le dictionnaire indique le score de cette coocurrence :

In [8]:
bigram_phrases.vocab[key_]

42

Lorsque l'instance de `Phrases` a été entraînée, elle peut concaténer les bigrams dans les phrases lorsque c'est pertinent.

### Conversion des `Phrases` en objet `Phraser`

`Phraser` est un alias pour `gensim.models.phrases.FrozenPhrases`, voir ici https://radimrehurek.com/gensim/models/phrases.html.

Le `Phraser` est une version *light* du `Phrases`, plus optimale pour transformer les phrases en concaténant les bigrams.

In [9]:
bigram_phraser = Phraser(phrases_model=bigram_phrases)

In [10]:
import time
t = time.time()
with open("../../data/sents_bigrams.txt", "w", encoding="utf-8") as f:
    for s in bigram_phraser[sentences]:
        f.write(" ".join(s) + "\n")
print("Écriture :", round(time.time() - t), "s")

Écriture : 22 s


In [11]:
from gensim.models.word2vec import LineSentence
corpus = LineSentence("../../data/sents_bigrams.txt")

Le `Phraser` est un objet qui convertit certains unigrams d'une liste en bigrams lorsqu'ils ont été identifiés comme pertinents.

### Extraction des trigrams

Nous répétons l'opération en envoyant cette fois la liste de bigrams afin d'extraire les trigrams.

In [11]:
trigram_phrases = Phrases(bigram_phraser[sentences])

MemoryError: 

In [ ]:
trigram_phraser = Phraser(phrases_model=trigram_phrases)

### Création d'un corpus d'unigrams, bigrams, trigrams

In [12]:
corpus = list(trigram_phraser[bigram_phraser[sentences]])

NameError: name 'trigram_phraser' is not defined

In [ ]:
print(corpus[:100])

## Entrainement d'un modèle Word2Vec sur ce corpus

In [13]:
t = time.time()
model_A = Word2Vec(corpus, vector_size=32, window=5, min_count=5, workers=4, epochs=5)
print("Modèle A :", round(time.time() - t), "s | vocabulaire :", len(model_A.wv))
model_A.save("../../data/w2v_A.model")

Modèle A : 48 s | vocabulaire : 95299


In [17]:
t = time.time()
model_B = Word2Vec(corpus, vector_size=32, window=10, min_count=10, workers=4, epochs=5)
print("Modèle B :", round(time.time() - t), "s | vocabulaire :", len(model_B.wv))
model_B.save("../../data/w2v_B.model")

Modèle B : 60 s | vocabulaire : 58527


In [18]:
t = time.time()
model_C = Word2Vec(corpus, vector_size=100, window=5, min_count=5, workers=4, epochs=10)
print("Modèle C :", round(time.time() - t), "s | vocabulaire :", len(model_C.wv))
model_C.save("../../data/w2v_C.model")

Modèle C : 138 s | vocabulaire : 95299


In [19]:
modeles = {"A": model_A, "B": model_B, "C": model_C}
for mot in ["ministre", "guerre", "bruxelles", "roi"]:
    print("=== ", mot)
    for nom, m in modeles.items():
        if mot in m.wv:
            voisins = [w for w, s in m.wv.most_similar(mot, topn=6)]
            print(nom, ":", ", ".join(voisins))
        else:
            print(nom, ": absent du vocabulaire")

===  ministre
A : president, conseiller, secretaire, ministere, gouverneur, bourgmestre
B : president, rapporteur, conseiller, ancien_ministre, membre, secretaire_general
C : le_ministre, ancien_ministre, secretaire_general, au_ministere, president, rapporteur
===  guerre
A : la_defense, grece, question, la_publication, pekin, crete
B : communication, crete, garnison, emis, la_convention, la_defense
C : la_guerre, pekin, greve, la_convention, communication, bulgarie
===  bruxelles
A : spa, des_ventes, ostende, schaerbeek, paris, namur
B : bruxolles, du_notaire, adjudications, siege_social, au_bulletin, du_congres
C : bruxolles, liege, schaerbeek, bruxeues, nivelles, hasselt
===  roi
A : college, prince, comte, docteur, colonel, tribunal
B : college, docteur, prince, messager, tribunal, redacteur
C : prince, chancelier, pape, tsar, souverain, baron


In [20]:
model = model_C
print(model.wv.similarity("ministre", "president"))
print(model.wv.similarity("roi", "prince"))
print(model.wv.similarity("bruxelles", "liege"))
print(model.wv.similarity("ministre", "roi"))

0.70547414
0.736055
0.56413037
0.4930963


In [21]:
for mot in ["gouvernement", "londres", "armee", "journal"]:
    print("===", mot)
    for w, s in model.wv.most_similar(mot, topn=8):
        print(f"  {w:25s} {s:.3f}")

=== gouvernement
  reglement                 0.771
  jury                      0.769
  comite                    0.754
  conseil                   0.751
  conseil_federal           0.737
  voeu                      0.733
  syndicat                  0.720
  principe                  0.710
=== londres
  berlin                    0.861
  paris                     0.791
  washington                0.736
  geneve                    0.730
  madrid                    0.705
  belgrade                  0.699
  rome                      0.694
  barcelone                 0.690
=== armee
  escadre                   0.766
  empire                    0.720
  invasion                  0.690
  evacuation                0.688
  organisation              0.687
  armee_allemande           0.684
  afrique                   0.684
  expedition                0.684
=== journal
  courrier                  0.709
  times                     0.702
  correspondant             0.695
  petit_journal             0.6

In [22]:
print(model.wv.most_similar(positive=["paris", "belgique"], negative=["france"], topn=5))

[('amsterdam', 0.5940631628036499), ('londres', 0.5921904444694519), ('geneve', 0.5814520716667175), ('marseille', 0.5742133259773254), ('hambourg', 0.5714117288589478)]


#### Remarque

Vous voyez ici que l'entrainement du modèle est parallélisé (sur 4 workers).

Lors qu'on parallélise l'entrainement du modèle, 4 modèles "séparés" sont entrainés sur environ un quart des phrases.

Ensuite, les résultats sont agrégés pour ne plus faire qu'un seul modèle.

On ne peut prédire quel worker aura quelle phrase, car il y a des aléas lors de la parallélisation (p. ex. un worker qui serait plus lent, etc.).

Du coup, les valeurs peuvent varier légèrement d'un entrainement à l'autre.

Mais, globalement, les résultats restent cohérents.

### Sauver le modèle dans un fichier

In [14]:
outfile = f"../../data/..."
model.save(outfile)

NameError: name 'model' is not defined

## Explorer le modèle

### Charger le modèle en mémoire

In [15]:
model = Word2Vec.load("../../data/...")

PermissionError: [Errno 13] Permission denied: '../../data/...'

### Imprimer le vecteur d'un terme

In [16]:
model.wv["ministre"]

NameError: name 'model' is not defined

### Calculer la similarité entre deux termes

In [ ]:
model.wv.similarity("ministre", "roi")

### Chercher les mots les plus proches d'un terme donné

In [ ]:
model.wv.most_similar("ministre", topn=10)

### Faire des recherches complexes à travers l'espace vectoriel

In [ ]:
print(model.wv.most_similar(positive=['paris', 'londres','allemagne'], negative=['france', 'grande_bretagne'], topn=1))